# Mesh-supervised PGD residual on GSO

Use a Colab T4 with a checkout of this repository. Set the paths below to your own Drive and checkout. The original Task 3 export, the full pilot run and its PGD predictions are prerequisites.

The default runs on the original 42/6/12 split. To compare 42 and 126 training objects on the fresh 24/42 validation and test sets, prepare the supplemental dataset and its cached PGD predictions using the [Task 4 instructions](../src/task4/README.md), then set `EXTRA_DATA` below.

In [ ]:
from pathlib import Path
from google.colab import drive
import json
import subprocess
import sys

drive.mount('/content/drive')

PROJECT = Path('/content/drive/MyDrive/MVP_Project')
REPO = Path('/content/L335_MiniProject')
BASE_RUN = None    # Path to the pilot run folder if there is more than one
EXTRA_DATA = None  # Path to the supplemental Task 3 split for the expanded comparison
OUTPUT = PROJECT / 'task4/mesh_residual'
MESH_CACHE = Path('/content/gso_mesh_cache')

data = PROJECT / 'task3/gso_task3_v1'
pgd_cache = PROJECT / 'task4/pgd_cache'
manifest = json.loads((data / 'manifest.json').read_text())
candidate_folder = f"candidates_{manifest['revision'][:8]}_{manifest['points']}_{manifest['seed']}_v1"
candidates = data.parent / candidate_folder

if BASE_RUN is None:
    pilot_runs = [
        p for p in (PROJECT / 'task4').glob('full_*')
        if (p / 'run.json').is_file()
        and (p / 'surface/texture_regions/mask_metadata.json').is_file()
    ]
    if len(pilot_runs) != 1:
        raise ValueError('Set BASE_RUN to a full pilot folder with run.json and texture-region masks.')
    BASE_RUN = pilot_runs[0]

required = [
    REPO / 'src/task4/surface_residual.py',
    BASE_RUN / 'run.json',
    BASE_RUN / 'surface/texture_regions/mask_metadata.json',
    candidates,
    pgd_cache,
]
if EXTRA_DATA is not None:
    required.extend([EXTRA_DATA / 'selection.json', EXTRA_DATA / 'expansion.json'])
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)

subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'trimesh==5.2.2',
                'point-cloud-utils'], check=True)

In [ ]:
command = [
    sys.executable, '-u', str(REPO / 'src/task4/surface_residual.py'),
    '--data', str(data), '--run', str(BASE_RUN), '--pgd-cache', str(pgd_cache),
    '--candidates', str(candidates), '--mesh-cache', str(MESH_CACHE),
]

if EXTRA_DATA is None:
    subprocess.run([*command, '--out', str(OUTPUT / 'pilot')], cwd=REPO, check=True)
else:
    expanded = OUTPUT / 'expanded'
    for count in (42, 126):
        subprocess.run([
            *command, '--extra', str(EXTRA_DATA), '--train-objects', str(count),
            '--out', str(expanded / f'train_{count}'),
        ], cwd=REPO, check=True)
    subprocess.run([
        sys.executable, str(REPO / 'src/task4/compare_gso_scale.py'), str(expanded),
    ], cwd=REPO, check=True)

In [ ]:
import pandas as pd

if EXTRA_DATA is None:
    summary = pd.read_csv(OUTPUT / 'pilot/summary.csv')
else:
    summary = pd.read_csv(OUTPUT / 'expanded/test_means_scaled.csv')
display(summary)